<a href="https://colab.research.google.com/github/vikramvundyala/python_AI-ML/blob/main/Hackathon_3_Voice_E_commerce_Ordering_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Certification in AIML
## A Program by IIIT-H and TalentSprint

In [ ]:
#@title Explanation Video
from IPython.display import HTML

HTML("""<video width="854" and height="480" controls>
  <source src="https://cdn.iiith.talentsprint.com/aiml/Experiment_related_data/Walkthrough/Hackathon_Voice_based.mp4" type="video/mp4">
</video>
""")

# Hackathon: Voice commands based E-commerce ordering system
The goal of the hackathon is to train your model on different types of voice data (such as studio data and your own team data) and able to place order based on user preferences.

## Grading = 40 Marks

### **Objectives:**

Stage 0 - Obtain Features from Audio samples

Stage 1 (22 Marks) - Define and train a CNN model on Studio data and deploy the model in the server

Stage 2 (18 Marks) - Collect your voice samples (team data) and refine the classifier trained on Studio_data. Deploy the model in the server.

## Dataset Description

The data contains voice samples of classes - Zero, One, Two, Three, Four, Five. Each class is denoted by a numerical label from 0 to 5.

The audio files collected in a Studio dataset contain very few noise samples and all the files are in wav format.

The audio files recorded for the studio are saved with the following naming convention:

● Class Representation + user_id + sample_ID (or noise + sample_ID)

> For example: The voice sample by the user b2 recorded “Zero”, it is saved as 0_b2_35.wav. Here 35 is sample ID, 2 is the user id and ‘0’ is the label of that sample.




In [ ]:
#@title Please run the setup to download the dataset

from IPython import get_ipython
ipython = get_ipython()

notebook= "Hackathon2 - Voice E-commerce Ordering System" #name of the notebook

def setup():
    ipython.magic("sx wget https://cdn.iiith.talentsprint.com/aiml/Hackathon_data/B17_studio_rev_data.zip")
    ipython.magic("sx unzip B17_studio_rev_data.zip ")
    print ("Setup completed successfully")

setup()

Setup completed successfully


In [34]:
import os
import sys
import glob
import torch
import librosa
import warnings
import numpy as np
import torch.nn as nn
from time import sleep
from torch import optim
import torch.nn.functional as F
from torch.autograd import Variable
warnings.filterwarnings('ignore')

## **Stage 0:** Obtain Features from Audio samples
---

### Generate features from an audio sample of '.wav' format
- Code is available to extract the features

In [45]:
# Caution: Do not change the default parameters
def get_features(filepath, sr=8000, n_mfcc=30, n_mels=128, frames = 15):
    # The following function contains code to produce features of the audio sample.
    y, sr = librosa.load(filepath, sr=sr)
    D = np.abs(librosa.stft(y))**2
    S = librosa.feature.melspectrogram(S=D)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)
    log_S = librosa.power_to_db(S,ref=np.max)
    features = librosa.feature.mfcc(S=log_S, n_mfcc=n_mfcc)
    if features.shape[1] < frames :
        features = np.hstack((features, np.zeros((n_mfcc, frames - features.shape[1]))))
    elif features.shape[1] > frames:
        features = features[:, :frames]

    # Find 1st order delta_mfcc
    delta1_mfcc = librosa.feature.delta(features, order=1)

    # Find 2nd order delta_mfcc
    delta2_mfcc = librosa.feature.delta(features, order=2)

    # Stacking delta_mfcc features in sequence horizontally (column wise)
    features = np.hstack((delta1_mfcc.flatten(), delta2_mfcc.flatten()))

    # Increase the dimension by inserting an axis along second dimension
    features = features.flatten()[:,np.newaxis]

    # Convert the numpy.ndarray to a Tensor object
    features = Variable(torch.from_numpy(features)).float()
    return features

All the voice samples needed for training are present in the folder `"studio_data"`

In [36]:
%ls

B17_studio_rev_data.zip  sample_data/  studio_data/


##**Stage 1**:  Define and train a CNN model on Studio data and deploy the model in the server

---


### a) Extract features of Studio data (4 Marks)

 Load 'Studio data' and extract mfcc features

 **Evaluation Criteria:**

 * Complete the code in the load_data function
 * The function should take path of the folder containing audio samples as input
 * It should return features of all the audio samples present in the specified folder into single array (list of lists or 2-d numpy array) and their respective labels should be returned too

In [46]:
def load_data(folder_path):
    #YOUR CODE HERE
  features_list = []
  labels = []
  for file in os.listdir(folder_path):
    file_path = os.path.join(folder_path, file)
    feature_tensor = get_features(file_path)
    print(feature_tensor.shape)
    features_list.append(feature_tensor.detach().numpy().flatten())
    labels.append(int(file[0]))
  return features_list, labels


Load data from studio_data folder for extracting all features and labels

In [47]:
studio_recorded_features, studio_recorded_labels = load_data('/content/studio_data')

torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([900, 1])
torch.Size([9

Use train_test_split for splitting the train and test data

In [48]:
from sklearn.model_selection import train_test_split

X = torch.tensor(studio_recorded_features, dtype=torch.float32)
y = torch.tensor(studio_recorded_labels, dtype=torch.long)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


Load the dataset with DataLoader
- Refer to [torch.utils.data.TensorDataset](https://pytorch.org/docs/stable/data.html#torch.utils.data.TensorDataset)
- Refer to [torch.utils.data.DataLoader](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader)

In [49]:
# YOUR CODE HERE for the DataLoader

import torch
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)



### b) Define your CNN architecture (4 Marks)

[Hint](https://pytorch.org/docs/stable/generated/torch.nn.Conv1d.html)

In [50]:
## Define your CNN Architecture
class Net(nn.Module):
    def __init__(self, feature_dim=900, num_classes=6):
        super(Net, self).__init__()

        # Convolutional layers
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=64, kernel_size=5)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(2)

        self.conv2 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=5)
        self.bn2 = nn.BatchNorm1d(128)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(2)

        self.conv3 = nn.Conv1d(in_channels=128, out_channels=256, kernel_size=5)
        self.bn3 = nn.BatchNorm1d(256)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool1d(2)

        self.dropout = nn.Dropout(0.25)

        # Compute input dimension for FC dynamically
        self.fc_input_dim = self._get_fc_input_dim(feature_dim)

        # Fully connected layers
        self.fc1 = nn.Linear(self.fc_input_dim, 128)
        self.relu_fc1 = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)
        self.relu_fc2 = nn.ReLU()
        self.fc3 = nn.Linear(64, num_classes)
        self.log_softmax = nn.LogSoftmax(dim=1)

    def _get_fc_input_dim(self, feature_dim):
        # Create a dummy tensor to calculate flattened size after conv/pool layers
        x = torch.zeros(1, 1, feature_dim)
        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu2(self.bn2(self.conv2(x))))
        x = self.pool3(self.relu3(self.bn3(self.conv3(x))))
        return x.numel()  # total elements after flattening

    def forward(self, x):
        # Conv layers
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu1(out)
        out = self.pool1(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu2(out)
        out = self.pool2(out)

        out = self.conv3(out)
        out = self.bn3(out)
        out = self.relu3(out)
        out = self.pool3(out)

        out = self.dropout(out)

        # Flatten
        out = out.view(out.size(0), -1)

        # Fully connected layers
        out = self.fc1(out)
        out = self.relu_fc1(out)
        out = self.fc2(out)
        out = self.relu_fc2(out)
        out = self.fc3(out)
        out = self.log_softmax(out)

        return out

In [51]:
# To run the training on GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [52]:
model = Net()
model = model.to(device)
print(model)

#criterion = # YOUR CODE HERE : Explore and declare loss function

criterion = nn.CrossEntropyLoss()

#optimizer = # YOUR CODE HERE : Explore on the optimizer and define with the learning rate

optimizer = optim.Adam(model.parameters(), lr=0.001)

Net(
  (conv1): Conv1d(1, 64, kernel_size=(5,), stride=(1,))
  (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu1): ReLU()
  (pool1): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv1d(64, 128, kernel_size=(5,), stride=(1,))
  (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu2): ReLU()
  (pool2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv1d(128, 256, kernel_size=(5,), stride=(1,))
  (bn3): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu3): ReLU()
  (pool3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.25, inplace=False)
  (fc1): Linear(in_features=27904, out_features=128, bias=True)
  (relu_fc1): ReLU()
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (relu_fc2): ReLU()
  (fc3): Linear(in_features=64

### c) Train and classify on the studio_data (3 Marks)

The goal here is to train the Model on voice samples collected in studio data and validate it continuously to calculate the loss and accuracy for the train dataset across each epoch.

Iterate over images in the train_loader and perform the following steps.

1. First, zero out the gradients using zero_grad()

2. Pass the data to the model. Convert the data to GPU before passing data  to the model

3. Calculate the loss using a Loss function

4. Perform Backward pass using backward() to update the weights

5. Optimize and predict by using the torch.max()

6. Calculate the accuracy of the train dataset


In [53]:
# YOUR CODE HERE. This will take time

# Record loss and accuracy of the train dataset

num_epochs = 20   # adjust based on dataset size

for epoch in range(num_epochs):
    model.train()  # set model in training mode

    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        # Move data to GPU if available
        inputs, labels = inputs.to(device), labels.to(device)

        # Ensure inputs are in correct shape [batch, channels, sequence_length]
        inputs = inputs.unsqueeze(1)   # add channel dimension: [batch, 1, 900]

        # 1️⃣ Zero out the gradients
        optimizer.zero_grad()

        # 2️⃣ Forward pass
        outputs = model(inputs)

        # 3️⃣ Compute loss
        loss = criterion(outputs, labels)

        # 4️⃣ Backward pass
        loss.backward()

        # 5️⃣ Update weights
        optimizer.step()

        # Accumulate loss
        running_loss += loss.item()

        # 6️⃣ Predictions
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    # Epoch metrics
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%")

Epoch [1/20], Loss: 1.3447, Accuracy: 46.12%
Epoch [2/20], Loss: 0.8880, Accuracy: 67.67%
Epoch [3/20], Loss: 0.6421, Accuracy: 78.04%
Epoch [4/20], Loss: 0.4765, Accuracy: 83.73%
Epoch [5/20], Loss: 0.3668, Accuracy: 86.87%
Epoch [6/20], Loss: 0.3106, Accuracy: 88.22%


KeyboardInterrupt: 

### d) Testing Evaluation for CNN model (3 Marks)

Evaluate model with the given test data

1. Transform and load the test images.

2. Pass the test data through the model (network) to get the outputs

3. Get the predictions from a maximum value using torch.max

4. Compare with the actual labels and get the count of the correct labels

5. Calculate the accuracy based on the count of correct labels

### **Expected testing accuracy is above 80%**

In [ ]:
# YOUR CODE HERE to test the model

model.eval()  # set model to evaluation mode

correct = 0
total = 0

with torch.no_grad():  # no gradients needed for evaluation
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)  # forward pass
        _, predicted = torch.max(outputs, 1)  # get predicted class

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = correct / total * 100
print(f"Test Accuracy: {accuracy:.2f}%")

### e) Save and download your model (2 Marks)

**Save your model trained on studio data**

* Save the state dictionary of the classifier (use pytorch only), It will be useful in
integrating model to the web application

 [Hint](https://pytorch.org/tutorials/beginner/saving_loading_models.html)

In [ ]:
### YOUR CODE HERE for saving the CNN model

Download your trained model using the code below
* Give the path of model file to download through the browser

In [ ]:
from google.colab import files
files.download('<model_file_path>')

### f) Deploy and evaluate your model trained on Studio Data in the server (6 Marks).

(This can be done on the day of the Hackathon once the login username and password provided by the mentors in the lab)

Deploy your model on the server, check the hackathon document (2-Server Access and File transfer For Voice based e-commerce ordering.pdf) for details.

To order product in user interface, go through the document (3-Hackathon_II Application Interface Documentation.pdf) for details.


**Evaluation Criteria: Four consecutive utterances should be predicted correctly by the model**

- There are two stages in the e-commerce ordering application    
    - Ordering Product
    - Selecting the e-commerce platform
- If both the stages are cleared as per the evaluation criteria you will get
complete marks Otherwise, you will see a reduction in the marks

## **Stage 2:** Collect your voice samples and refine the classifier trained on studio_data and Team_data
---

### a) Collect your Team Voice Samples and extract features (6 Marks)

(This can be done on the day of the Hackathon once the login username and password is given by mentors in the lab)

* In order to collect the team data, ensure the server is active (2-Server Access and File transfer For Voice based e-commerce ordering.pdf)

* Refer document "3-Hackathon_II Application Interface Documentation.pdf" for collecting your team voice samples. These will get stored in your server

**Evaluation Criteria:**
* Load 'Team_data' and extract features
* Combine features of team data with the extracted features of studio data
* Split the combined features into train and test data
* Load the dataset with DataLoader

In [ ]:
!mkdir team_data

In [ ]:
# Replace <YOUR_GROUP_ID> with your Username given in the lab
!wget -r -A .wav https://aiml-sandbox1.talentsprint.com/audio_recorder/<YOUR_GROUP_ID>/team_data/ -nH --cut-dirs=100  -P ./team_data

In [ ]:
# YOUR CODE HERE to Load data from teamdata folder for extracting all features and labels

In [ ]:
# Combine the features of all voice samples (studio_data and teamdata)
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE to split the combined features into train and test data (Hint: Use train_test_split)

In [ ]:
# YOUR CODE HERE to load the dataset with DataLoader

### b) Classify and download the model (6 Marks)

The goal here is to train and test your model on all voice samples collected in studio and team data

**Evaluation Criteria:**
* Refine your classifier (if needed)
* Train your model on the extracted train data
* Test your model on the extracted test data
* Save and download the trained model

### **Expected testing accuracy is above 80%**

In [ ]:
# YOUR CODE HERE for refining your classifier (if needed)

In [ ]:
# YOUR CODE HERE to train your model

# Record loss and accuracy of the train dataset

In [ ]:
# YOUR CODE HERE to test your model

**Save your trained model**

* Save the state dictionary of the classifier (use pytorch only), It will be useful in
integrating model to the web application

 [Hint](https://pytorch.org/tutorials/beginner/saving_loading_models.html)

In [ ]:
### YOUR CODE HERE for saving the CNN model

Download your trained model using the code below
* Give the path of model file to download through the browser

In [ ]:
from google.colab import files
files.download('<model_file_path>')

### c) Deploy and evaluate your model trained on Studio Data + Team Data in the server (6 Marks).

(This can be done on the day of the Hackathon once the login username and password provided by the mentors in the lab)

Deploy your model on the server, check the hackathon document (2-Server Access and File transfer For Voice based e-commerce ordering.pdf) for details.

To order product in user interface, go through the document (3-Hackathon_II Application Interface Documentation.pdf) for details.


**Evaluation Criteria: Four consecutive utterances should be predicted correctly by the model**

- There are two stages in the e-commerce ordering application    
    - Ordering Product
    - Selecting the e-commerce platform
- If both the stages are cleared as per the evaluation criteria you will get
complete marks Otherwise, you will see a reduction in the marks